# 8. Minimum vertex cover

Lecture: *Week 5 - Mixed Integer Programming*. This model is not on the slides: it is a preview of Week 06
(demo 4, security cameras). Run it after demo 3, the fire stations.

**The problem.** A graph has six vertices and five edges: 1–2, 2–6, 3–4, 4–5 and 5–6 (a path
1–2–6–5–4–3). Choose as few vertices as possible so that every edge has at least one chosen end. $x_i = 1$ if
vertex $i$ is chosen:

minimise $\sum_{i=1}^{6} x_i$ subject to $x_u + x_v \ge 1$ for every edge $(u, v)$, $x_i \in \{0, 1\}$.

It is a set covering problem in which every constraint has exactly two variables (the two ends of an edge).
In the code, vertices 1–6 are `x[0]` … `x[5]`. The constraints are named `city1` … `city5` (the names were
copied from the fire-station demo), but each one is an **edge**.

**How to run it.** Run the first code cell first (it installs Pyomo and the HiGHS solver; the first time takes about a minute), then *Run → Run All Cells*, or run the cells one by one while you read.

**Expected result.** **3 vertices**. The browser shows vertices 2, 4 and 5 (`x[1]`, `x[3]`, `x[4]`); other sets
of three, such as 2, 4 and 6, are also optimal, so another solver may show one of those.

*Original script: `5 Source Files/Python/minimum_vertex_cover.py`. Runs in the browser with HiGHS instead of GLPK; the model and the optimal value are unchanged.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

Load Pyomo's modelling tools.

In [ ]:
from pyomo.environ import *

A *concrete model* is the container for the problem.

In [ ]:
# Create a concrete model
model = ConcreteModel()

One binary variable per vertex: chosen (1) or not (0).

In [ ]:
# Define the binary variables: x[i] = 1 if vertex i+1 is in the cover
model.x = Var(range(6), within=Binary)

The objective: the number of chosen vertices, minimised.

In [ ]:
# Define the objective function
model.obj = Objective(expr=sum(model.x[i] for i in range(6)), sense=minimize)

One constraint per edge: at least one of its two ends is chosen.

In [ ]:
# Define the inequality constraints: one per edge (the names city1-city5 are edges, not cities)
model.city1 = Constraint(expr=model.x[0] + model.x[1] >= 1)  # edge 1-2
model.city2 = Constraint(expr=model.x[1] + model.x[5] >= 1)  # edge 2-6
model.city3 = Constraint(expr=model.x[2] + model.x[3] >= 1)  # edge 3-4
model.city4 = Constraint(expr=model.x[3] + model.x[4] >= 1)  # edge 4-5
model.city5 = Constraint(expr=model.x[4] + model.x[5] >= 1)  # edge 5-6

HiGHS is the solver; unlike GLPK it runs in the browser.

In [ ]:
# Create a solver
solver = SolverFactory('appsi_highs')

`pprint()` prints the model with the data filled in: check that there is one constraint per edge.

In [ ]:
# Display the model
model.pprint()

Solve. In the browser HiGHS prints its log here even with `tee=False` (on a desktop it stays quiet); the *Solving report* at the end should say *Status Optimal*.

In [ ]:
# Solve the model
solver.solve(model)

`display()` prints which vertices are chosen and the objective (the size of the cover).

In [ ]:
# Display the results
model.display()

## What do you notice?

Why can two vertices never cover these five edges?

<details><summary>Answer</summary>

The edges form a path, so every vertex touches at most two edges. Two vertices cover at most four edges, and
there are five. Three vertices are enough, so 3 is the minimum.
</details>